# Multi Documents connecting to RAG


In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq


C:\Users\jasme\anaconda3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup & reconnect 

In [2]:
# Load environment
env_path = Path(r"C:\Users\jasme\Documents\rag-knowledge-assistant\.env")
load_dotenv(dotenv_path=env_path, override=True)

# Reconnect to ChromaDB (persistent)
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

persistent_client = chromadb.PersistentClient(
    path=r"C:\Users\jasme\Documents\rag-knowledge-assistant\chroma_db"
)

collection = persistent_client.get_collection(
    name="rag_paper",
    embedding_function=sentence_transformer_ef
)

# Reconnect to Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Text splitter (same settings as Day 2)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print(f"✅ Connected! Current chunks in collection: {collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2835.42it/s]


✅ Connected! Current chunks in collection: 229


###  Process PDF

In [3]:
def process_pdf(pdf_path, source_name):
    """Extract, chunk, and attach metadata for each page of a PDF."""
    reader = PdfReader(pdf_path)
    
    all_chunks = []
    all_metadatas = []
    
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        page_chunks = splitter.split_text(text)
        
        for chunk in page_chunks:
            all_chunks.append(chunk)
            all_metadatas.append({
                "source": source_name,
                "page": page_num + 1
            })
    
    return all_chunks, all_metadatas

print("✅ process_pdf() ready")

✅ process_pdf() ready


### Second Documents uploading

In [4]:
chunks2, metadatas2 = process_pdf(
    r"C:\Users\jasme\Documents\rag-knowledge-assistant\data\test_for_rag_part_2.pdf",
    "paper2.pdf"
)

ids2 = [f"paper2_chunk_{i}" for i in range(len(chunks2))]

collection.add(
    documents=chunks2,
    metadatas=metadatas2,
    ids=ids2
)

print(f"✅ Added {len(chunks2)} chunks from paper2.pdf")
print(f"Total chunks now: {collection.count()}")

✅ Added 136 chunks from paper2.pdf
Total chunks now: 229


In [5]:
# Get all 93 original chunks (they have IDs chunk_0 to chunk_92)
old_ids = [f"chunk_{i}" for i in range(93)]
old_data = collection.get(ids=old_ids)

# Build metadata for each — all from "test_for_rag.pdf"
old_metadatas = [{"source": "test_for_rag.pdf", "page": "unknown"} for _ in old_ids]

# Upsert — this updates existing entries with metadata added
collection.upsert(
    ids=old_ids,
    documents=old_data['documents'],
    metadatas=old_metadatas
)

print(f"✅ Updated {len(old_ids)} old chunks with metadata")

✅ Updated 93 old chunks with metadata


In [6]:
def retrieve_chunks(query, n_results=3):
    results = collection.query(query_texts=[query], n_results=n_results)
    return results['documents'][0], results['metadatas'][0]

def build_prompt(query, chunks):
    context = "\n\n".join(chunks)
    return f"""Answer the question based only on the context below.
If the context doesn't contain enough information, say "I don't have enough information to answer this."

Context:
{context}

Question: {query}

Answer:"""

def generate_answer(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers based strictly on provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

def rag_query_with_sources(question, n_results=3):
    chunks, metadatas = retrieve_chunks(question, n_results)
    prompt = build_prompt(question, chunks)
    answer = generate_answer(prompt)
    
    sources = []
    for c, m in zip(chunks, metadatas):
        # Handle chunks with no metadata (original 93 chunks)
        if m is None:
            m = {}
        sources.append({
            "text": c[:150] + "...",
            "source": m.get("source", "paper1.pdf (legacy)"),
            "page": m.get("page", "?")
        })
    
    return {"question": question, "answer": answer, "sources": sources}

print("✅ RAG functions updated")

✅ RAG functions updated


### RAG functions

In [8]:
# Ask something specific to paper2 — adjust based on what paper2 is about
result = rag_query_with_sources("What is this paper about?")  
print(f"Answer: {result['answer']}\n")
print("Sources:")
for s in result['sources']:
    print(f"  - {s['source']}, page {s['page']}: {s['text']}")

Answer: I don't have enough information to answer this. The context provided only discusses parameters and configurations related to a PDF processing pipeline, but it does not mention the topic or subject of the paper.

Sources:
  - test_for_rag.pdf, page unknown: and recall of header/footer removal.
•min_samples : This parameter represents the minimum number of samples in a
neighborhood for a point to be consid...
  - paper2.pdf, page 9: intensive document retrieval.arXiv preprint arXiv:2509.24869, 2025.
Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin, ...
  - test_for_rag.pdf, page unknown: documents of varying lengths.
•eps(epsilon): This parameter defines the maximum distance between two samples for one
to be considered as in the neighb...
